In [0]:
import pandas as pd
from pyspark.sql.functions import col, count, when, avg
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
# =============================================================================
# LEITURA DE DADOS 
# =============================================================================

# FONTE: https://www.kaggle.com/datasets/waleedfaheem/airline-route-profitability-and-cost-analysis

TABLE_NAME = "default.airline_route_profitability"

try:

    # -------------------------------------------------------------------------
    # Leitura da tabela no catálogo do Databricks
    # -------------------------------------------------------------------------

    df = spark.table(TABLE_NAME)

    print("✅ Tabela '{TABLE_NAME}' carregada com sucesso!")

    # -------------------------------------------------------------------------
    # Informações iniciais
    # -------------------------------------------------------------------------
    
    total_rows = df.count()
    print(f"Total de registros: {total_rows}")

    display(df.limit(5))

except Exception as e:
    print(f"❌ Erro ao carregar a tabela '{TABLE_NAME}'")
    print(f"Detalhes do erro: {e}")

In [0]:
df.printSchema()



In [0]:
display(df.describe())

In [0]:
print(df.columns)

In [0]:
print(f"Linhas totais: {total_rows}")
print(f"Linhas distintas: {df.distinct().count()}")

In [0]:
from pyspark.sql.functions import col, count, when

display(
    df.select([
        count(
            when(col(c).isNull(), c)
        ).alias(c)
        for c in df.columns
    ])
)

In [0]:
from pyspark.sql.functions import when

df = df.withColumn(
    "Profitability",
    when(df.Profit_Margin >= 20, "High")
    .when(df.Profit_Margin >= 10, "Medium")
    .when(df.Profit_Margin >= 0, "Low")
    .otherwise("Loss")
)

In [0]:
df = df.withColumn(
    "Profit_Per_Hour",
    when(df.Flight_Hours > 0,
    df.Profit / df.Flight_Hours)
)

In [0]:
df = df.withColumn(
    "Revenue_Per_Passenger",
    when(df.Passengers > 0,
    df.Total_Revenue / df.Passengers)
)

In [0]:
df = df.withColumn(
    "Cost_Per_Passenger",
    when(df.Passengers > 0,
    df.Total_Cost / df.Passengers)
)

In [0]:
df = df.withColumn(
    "Profit_Per_Passenger",
    when(df.Passengers > 0,
    df.Profit / df.Passengers)
)

In [0]:
df.write.mode("overwrite").saveAsTable(
    "default.airline_route_profitability_enriched"
)

In [0]:
display(df.groupBy("Route").avg("Profit").orderBy(col("avg(Profit)").desc()))

In [0]:
display(df.groupBy("Route").avg("Profit_Margin").orderBy(col("avg(Profit_Margin)").desc()).limit(10))

In [0]:
display(df.groupBy("Aircraft_Type").avg("Profit").orderBy(col("avg(Profit)").desc()))


In [0]:
display(df.groupBy("Demand_Level").avg("Profit_Margin"))

In [0]:
display(df.groupBy("Season").agg({"Profit":"avg",
                                  "Passengers":"avg"}))

In [0]:
display(df.groupBy("Route_Category").agg(
avg((col("Fuel_Cost") / col("Total_Cost")) * 100)
)
)


In [0]:
display(df.groupBy("Profitability").agg(
    avg((col("Fuel_Cost") / col("Total_Cost")) * 100)
)
)

In [0]:
display(df.groupBy("Profitability").avg("Load_Factor").orderBy(col("avg(Load_Factor)").desc()))

In [0]:
display(df.groupBy("Profitability").avg("Revenue_Per_Passenger").orderBy(col("avg(Revenue_Per_Passenger)").desc()))

In [0]:
display(
    df.groupBy("Profitability")
      .avg(
          "Profit",
          "Load_Factor",
          "Revenue_Per_Passenger",
          "Cost_Per_Passenger",
          "Profit_Per_Hour"
      ).orderBy(col("avg(Profit)").desc()
)
)

In [0]:
display(df.groupBy("Aircraft_Type").avg("Profit_Per_Hour", "Profit_Margin", "Load_Factor").orderBy(col("avg(Profit_Per_Hour)").desc()))

In [0]:
display(df.groupBy("Aircraft_Type").avg("Revenue_Per_Passenger", "Cost_Per_Passenger", "Profit_Per_Passenger").orderBy(col("avg(Revenue_Per_Passenger)").desc())
)

In [0]:
display(
    df.groupBy("Route_Category", "Aircraft_Type")
      .avg("Profit_Margin")
      .orderBy(
          col("Route_Category"),
          col("avg(Profit_Margin)").desc()
      )
)

In [0]:
display(
    df.groupBy("Route_Category")
      .avg(
          "Revenue_Per_Passenger",
          "Profit_Per_Hour",
          "Profit_Margin"
      )
)

In [0]:
# =============================================================================
# MATRIZ DE CORRELAÇÃO
# =============================================================================

# Colunas com métricas de negócio

business_cols = [
    "Load_Factor",
    "Flight_Hours",
    "Passengers",
    "Total_Revenue",
    "Total_Cost",
    "Profit",
    "Profit_Margin",
    "Profit_Per_Hour"
]

# Converter para Pandas e calcular a matriz de correlação	

business_corr = (
    df
    .select(business_cols)
    .toPandas()
    .corr()
)


# Plotar a matriz de correlação
plt.figure(figsize=(8, 6))

sns.heatmap(
    business_corr,
    annot=True,
    fmt=".2f",
    cmap="RdYlBu_r",
    center=0
)

plt.title("Business Metrics Correlation")
plt.tight_layout()
plt.show()
# =============================================================================
# 

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import avg

# Lista das colunas de custo
cost_columns = [
    "Fuel_Cost",
    "Maintenance_Cost",
    "Crew_Cost",
    "Depreciation_Cost",
    "Insurance_Cost",
    "Airport_Fees",
    "Catering_Cost",
    "Handling_Cost",
    "Navigation_Fees",
    "Sales_Distribution_Cost",
    "Passenger_Service_Cost",
    "Overhead_Cost",
    "Marketing_Cost",
    "IT_Systems_Cost"
]

# Calcular o percentual médio de cada custo em relação ao custo total
results = []

for cost_col in cost_columns:
    
    avg_pct = (
        df
        .select(((df[cost_col] / df["Total_Cost"]) * 100).alias("pct"))
        .agg(avg("pct").alias("avg_pct"))
        .collect()[0]["avg_pct"]
    )

    results.append(
        Row(
            Cost_Category=cost_col,
            Avg_Percentage=round(avg_pct, 2)
        )
    )

# Criar DataFrame Spark
cost_pct_df = spark.createDataFrame(results)

# Ordenar do maior para o menor percentual
cost_pct_df = cost_pct_df.orderBy(
    col("Avg_Percentage").desc()
)

cost_pct_df.write.mode("overwrite").saveAsTable(
    "default.cost_structure_analysis"
)

display(cost_pct_df)